<a href="https://colab.research.google.com/github/psiperez/budas_trade/blob/main/barra1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def calculate_backtest_barra1():
    print("=" * 70)
    print("BACKTEST BARRA1 - RESULTADOS")
    print("=" * 70)

    # 1. Download Data - Ajustado para os últimos 60 dias para garantir disponibilidade de dados 1h
    end_date = datetime.now()
    start_date = end_date - timedelta(days=58)

    data_1h = yf.download('GC=F', start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='15m', progress=False)

    if data_1h.empty:
        return "Dados não encontrados. Tente um ativo diferente ou um período mais curto."

    if isinstance(data_1h.columns, pd.MultiIndex):
        data_1h.columns = data_1h.columns.get_level_values(0)

    # 2. Preparar dados simulados de M15
    df_list = []
    prev_close = data_1h.iloc[0]['Open']
    for idx, row in data_1h.iterrows():
        for i in range(4):
            factor = (i + 1) / 4.0
            candle_time = idx + timedelta(minutes=15 * i)
            candle_open = prev_close
            range_p = float(row['High']) - float(row['Low'])
            candle_high = float(row['Open']) + range_p * (factor * 0.8 + np.random.uniform(0, 0.2))
            candle_low = float(row['Open']) - range_p * (0.2 + np.random.uniform(0, 0.2))
            candle_high = max(float(candle_high), float(candle_low) + 0.1)
            candle_close = float(row['Open']) + range_p * factor if i < 3 else float(row['Close'])
            prev_close = candle_close
            df_list.append({'dt': candle_time, 'Open': candle_open, 'High': candle_high, 'Low': candle_low, 'Close': candle_close})

    df = pd.DataFrame(df_list).set_index('dt')

    # 3. Simulação simplificada
    balance = 100000.0
    trades = []
    positions = []

    for idx, row in df.iterrows():
        for p in positions[:]:
            hit_sl = (p['dir'] == 1 and row['Low'] <= p['sl']) or (p['dir'] == -1 and row['High'] >= p['sl'])
            hit_tp = (p['dir'] == 1 and row['High'] >= p['tp']) or (p['dir'] == -1 and row['Low'] <= p['tp'])
            if hit_sl or hit_tp:
                res = (p['sl'] if hit_sl else p['tp']) - p['entry']
                profit = res * p['dir'] * p['lot'] * 100
                balance += profit
                trades.append(profit)
                positions.remove(p)

        if len(positions) == 0 and np.random.random() < 0.05:
            dir = 1 if row['Close'] > row['Open'] else -1
            entry = row['Close']
            sl = entry - (5 * dir)
            tp = entry + (10 * dir)
            positions.append({'entry': entry, 'sl': sl, 'tp': tp, 'dir': dir, 'lot': 1.0})

    print(f"\nPeríodo: {start_date.date()} até {end_date.date()}")
    print(f"Saldo Final: ${balance:,.2f}")
    print(f"Total de Trades: {len(trades)}")
    print(f"Lucro Líquido: ${sum(trades):,.2f}")

calculate_backtest_barra1()

BACKTEST BARRA1 - RESULTADOS


/tmp/ipykernel_5785/1456228476.py:15: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data_1h = yf.download('GC=F', start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='15m', progress=False)



Período: 2026-01-18 até 2026-03-17
Saldo Final: $-199,500.00
Total de Trades: 668
Lucro Líquido: $-299,500.00
